# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, their fields, and their `@id` values using the Croissant schema. Listing out record sets and field meta-information by their `@id`s for consistent referencing.

In [ ]:
# Examine available record sets and their fields
# The Croissant metadata exposes all record sets in metadata.record_sets.
# Each record set has an @id, and a list of fields (referenced by @id).

record_sets_info = []
for record_set in getattr(metadata, 'record_sets', []):
    rs_id = getattr(record_set, '@id', None)
    rs_name = getattr(record_set, 'name', None)
    print(f"RecordSet: {rs_id} | Name: {rs_name}")

    print("  Fields:")
    for field in getattr(record_set, 'fields', []):
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', None)
        field_type = getattr(field, 'data_type', None)
        print(f"    - {field_id} | Name: {field_name} | Type: {field_type}")

    record_sets_info.append({'id': rs_id, 'name': rs_name})

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Choose record sets by their `@id`s for extraction. For this dataset, most relevant data is likely in the main table.
record_set_ids = [rs['id'] for rs in record_sets_info if rs['id'] is not None]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns from the first available record set
if record_set_ids:
    selected_rs_id = record_set_ids[0]
    print(f"Columns for record set `{selected_rs_id}`:")
    print(dataframes[selected_rs_id].columns.tolist())
    display(dataframes[selected_rs_id].head())
else:
    print("No record sets found in metadata.")

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group the data using field `@id` for referencing columns, as explored above.

Steps:
- Choose a numeric field (`@id`) for filtering and normalization
- Filter the DataFrame based on a threshold value
- Normalize the numeric field
- Optionally group data by a group field (`@id`)

In [ ]:
import numpy as np

# If we know the main record set and field ids, set them here. Otherwise, infer from the record set and field overview.
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]

    # Let's try to automatically find the first numeric field (float or int data_type from schema)
    numeric_field_id = None
    group_field_id = None
    for record_set in getattr(metadata, 'record_sets', []):
        if getattr(record_set, '@id', None) == main_record_set_id:
            for field in getattr(record_set, 'fields', []):
                dtype = getattr(field, 'data_type', None)
                if dtype in ['schema:Float', 'schema:Integer', 'schema:Number'] and numeric_field_id is None:
                    numeric_field_id = getattr(field, '@id', None)
                # Pick first string (categorical) field as group_field
                if dtype in ['schema:Text'] and group_field_id is None:
                    group_field_id = getattr(field, '@id', None)

    # If the field is found, demonstrate EDA
    if numeric_field_id is not None and numeric_field_id in df.columns:
        threshold = np.nanmean(df[numeric_field_id])  # Use mean as a threshold for demo
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized values for field `{numeric_field_id}`:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped and averaged (numeric only) by `{group_field_id}`:")
            display(grouped_df.head())
        else:
            print("No suitable categorical (group) field found for grouping.")
    else:
        print("No suitable numeric field found for filtering and normalization.")
else:
    print("No main record set available for EDA.")

## 5. Visualization
Visualize the distribution of the main numeric field and its normalized values. Use plotting libraries such as matplotlib or seaborn if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if the numeric field and data exist
if 'filtered_df' in locals() and numeric_field_id is not None and numeric_field_id in filtered_df:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col in filtered_df:
        sns.histplot(filtered_df[norm_col].dropna(), bins=20, kde=True)
        plt.title(f'Normalized {numeric_field_id}')
        plt.xlabel(norm_col)

    plt.tight_layout()
    plt.show()
else:
    print("No filtered data or numeric field to plot.")

## 6. Conclusion
In this notebook, we've loaded Croissant-formatted metadata and records using the `mlcroissant` library, explored available record sets and fields via their `@id`, and performed simple filtering, normalization, grouping, and visualization using these canonical identifiers. This approach ensures reproducibility and clarity across FAIR data packages and analyses.